# CokeSense — full replication notebook
### Every table, figure, and audited number of the ICDCN 2027 manuscript, from one workbook

This notebook reproduces the paper end to end on the plant record
`Coke_Oven_Data_Set.xlsx` (1,251 daily rows, Dec 2012 - Mar 2023): the
sixteen-model benchmark under random and chronological protocols, the
replicate-noise ceiling (Proposition 3.1), the hybrid sensor, the three
conformal calibration regimes including the rolling scheme, the ablation and
importance studies, the edge budget, the six publication figures with the
exact filenames the manuscript expects, and the deployment artifacts used by
the Streamlit dashboard.

**Runtime.** About 35 minutes on a free Colab CPU with `FULL_RUN = True`
(the paper setting); about 6 minutes with `FULL_RUN = False` for a quick
structural pass. Figures land in `figs/`, models in `models/`, and a zip of
both is offered at the end.

## 0 · Setup
Pinned installs (only what Colab lacks), imports, plotting style, seed.

In [ ]:
import os, sys, subprocess, importlib
PINS = {"scikit-learn": "1.8.0", "xgboost": "3.3.0", "openpyxl": "3.1.5"}
for pkg, ver in PINS.items():
    mod = pkg.replace("-", "_").replace("scikit_learn", "sklearn")
    try:
        m = importlib.import_module(mod)
        ok = getattr(m, "__version__", "?") == ver
    except ImportError:
        ok = False
    if not ok:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"{pkg}=={ver}"], check=True)
import numpy as np, pandas as pd, joblib, json, time, pickle
from math import ceil
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import sklearn, xgboost
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| numpy", np.__version__)

matplotlib.rcParams.update({
    "font.size": 7.2, "font.family": "serif", "mathtext.fontset": "stix",
    "axes.labelsize": 7.5, "axes.titlesize": 8, "xtick.labelsize": 6.6,
    "ytick.labelsize": 6.6, "legend.fontsize": 6.4, "lines.linewidth": 1.0,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.4,
    "axes.spines.top": False, "axes.spines.right": False})
COL = {"CRI": "#B34700", "CSR": "#1F5FA8"}
os.makedirs("figs", exist_ok=True); os.makedirs("models", exist_ok=True)
RANDOM_STATE = 42

## 1 · Configuration
Everything adjustable lives here. `FULL_RUN = True` reproduces the paper
exactly (5x3 repeated CV, 50-permutation importance, 300 timed latency calls);
`False` gives a fast structural pass whose numbers will differ slightly.

In [ ]:
FULL_RUN      = os.environ.get("COKESENSE_FULL", "1") == "1"
TARGETS       = ["CRI", "CSR"]
TARGET_BOUNDS = {"CRI": (10.0, 40.0), "CSR": (45.0, 78.0)}
RAW_FEATURES  = ["Coke_WM", "Coke_Ash", "Coke_VM", "Coke_FC",
                 "M40", "M10", "Coke_+80mm", "Coke_-25mm"]
FINAL_FEATURES = RAW_FEATURES + ["FC_Ash_ratio", "Strength_idx", "Ash_VM"]
W             = 100                       # rolling conformal window
N_REPEATS     = 3 if FULL_RUN else 1      # repeated-CV repeats
N_PERM        = 50 if FULL_RUN else 10    # permutation-importance repeats
N_TIMED       = 300 if FULL_RUN else 40   # latency calls
DATA_CANDIDATES = ["data/Coke_Oven_Data_Set.xlsx", "Coke_Oven_Data_Set.xlsx",
                   "../data/Coke_Oven_Data_Set.xlsx"]
DATA_URL = ("https://raw.githubusercontent.com/parichay-india/CokeSense/"
            "main/data/Coke_Oven_Data_Set.xlsx")
print("FULL_RUN =", FULL_RUN)

## 2 · Data
The loader looks for the workbook next to the notebook (the repository
layout), then tries the repository raw URL, and finally offers a Colab upload.
Parsing follows the paper: `Coke` sheet, banner rows skipped, duplicate
determinations averaged, physically impossible targets blanked, repeated dates
collapsed by mean.

In [ ]:
def _find_workbook():
    for p in DATA_CANDIDATES:
        if os.path.exists(p):
            return p
    try:
        import urllib.request
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(DATA_URL, "data/Coke_Oven_Data_Set.xlsx")
        return "data/Coke_Oven_Data_Set.xlsx"
    except Exception as e:
        print("download failed:", e)
    try:
        from google.colab import files          # noqa: Colab only
        up = files.upload()
        return list(up)[0]
    except Exception:
        raise FileNotFoundError("place Coke_Oven_Data_Set.xlsx beside this notebook")

DATA_PATH = _find_workbook()
raw = pd.read_excel(DATA_PATH, sheet_name="Coke", header=None)

def col(i, start=5):
    return pd.to_numeric(raw.iloc[start:, i], errors="coerce").reset_index(drop=True)

def avg2(i, j):
    return pd.concat([col(i), col(j)], axis=1).mean(axis=1)

df = pd.DataFrame({
    "Date": pd.to_datetime(raw.iloc[5:, 0], errors="coerce").reset_index(drop=True),
    "Coke_WM": col(3), "Coke_Ash": col(6), "Coke_VM": col(9), "Coke_FC": col(12),
    "M40": col(15), "M10": col(18),
    "Coke_+80mm": avg2(19, 20), "Coke_-25mm": avg2(21, 22),
    "CRI": avg2(23, 24), "CSR": avg2(25, 26)})
df = df.dropna(subset=["Date"])
df["Date"] = df["Date"].dt.normalize()
df = df.groupby("Date", as_index=False).mean()
for t, (lo, hi) in TARGET_BOUNDS.items():
    df.loc[(df[t] < lo) | (df[t] > hi), t] = np.nan
df = (df.dropna(subset=TARGETS, how="all")
        .sort_values("Date").reset_index(drop=True))
df["FC_Ash_ratio"] = df["Coke_FC"] / df["Coke_Ash"].replace(0, np.nan)
df["Strength_idx"] = df["M40"] - df["M10"]
df["Ash_VM"] = df["Coke_Ash"] * df["Coke_VM"]

REPS = pd.DataFrame({
    "Date": pd.to_datetime(raw.iloc[5:, 0], errors="coerce").reset_index(drop=True),
    "CRI_a": col(23), "CRI_b": col(24),
    "CSR_a": col(25), "CSR_b": col(26)}).dropna(subset=["Date"]).reset_index(drop=True)
REPS["Date"] = REPS["Date"].dt.normalize()

def per_target(t):
    """Per-target modelling frame: exactly coke_data.get_Xy plus split indices."""
    d = df.dropna(subset=[t]).sort_values("Date").reset_index(drop=True)
    nt = len(d)
    return d, d[FINAL_FEATURES], d[t].values, int(round(.6 * nt)), int(round(.8 * nt))

print(f"rows (union frame): {len(df)}  span: "
      f"{df.Date.iloc[0].date()} .. {df.Date.iloc[-1].date()}")
N_T, I80_T = {}, {}
for t in TARGETS:
    d_, X_, y_, i60_, i80_ = per_target(t)
    N_T[t], I80_T[t] = len(y_), i80_
    print(f"{t}: n={len(y_)}  mean {y_.mean():.2f}  sd {y_.std(ddof=1):.3f}  "
          f"chrono train {i80_} / test {len(y_) - i80_} "
          f"(from {d_.Date.iloc[i80_].date()})")

## 3 · The two structural facts
Both indices live in narrow bands for a decade, and those bands tighten and
shift in the final years: the test-window variance collapses by a factor of
three to four relative to the training years. **Figure 2 of the paper.**

In [ ]:
sd1 = {}
for t in TARGETS:                     # robust single-replicate sd (bounds-cleaned)
    lo_b, hi_b = TARGET_BOUNDS[t]
    a = REPS[f"{t}_a"].where(REPS[f"{t}_a"].between(lo_b, hi_b))
    b = REPS[f"{t}_b"].where(REPS[f"{t}_b"].between(lo_b, hi_b))
    both = np.isfinite(a) & np.isfinite(b)
    dd = (a[both] - b[both]).values
    sd1[t] = float(np.abs(dd).mean() * np.sqrt(np.pi / 2) / np.sqrt(2))

for t in TARGETS:
    _, _, y, _, cut = per_target(t)
    vt, vs = np.var(y[:cut], ddof=1), np.var(y[cut:], ddof=1)
    print(f"{t}: var(train)={vt:.3f}  var(test)={vs:.3f}  collapse x{vt/vs:.1f}")

fig, axes = plt.subplots(2, 1, figsize=(7.0, 2.9), sharex=True)
for ax, t in zip(axes, TARGETS):
    d, _, yv, i60_, i80_ = per_target(t)
    dts = d["Date"].values
    ax.plot(dts, yv, ".", ms=1.4, color=COL[t], alpha=.55)
    mu = yv.mean()
    ax.axhline(mu, color="k", lw=.6, ls="--", alpha=.7)
    ax.axhspan(mu - sd1[t]/np.sqrt(2), mu + sd1[t]/np.sqrt(2), color="k", alpha=.08, lw=0)
    ax.axvspan(dts[i60_], dts[i80_], color="#888", alpha=.12, lw=0)
    ax.axvspan(dts[i80_], dts[-1], color="#c33", alpha=.10, lw=0)
    ax.set_ylabel(t)
    ax.annotate("calibration", (dts[i60_ + (i80_ - i60_)//2], ax.get_ylim()[1]),
                ha="center", va="top", fontsize=6, color="#555")
    ax.annotate("test", (dts[i80_ + (len(yv) - i80_)//2], ax.get_ylim()[1]),
                ha="center", va="top", fontsize=6, color="#a22")
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].set_xlabel("date")
fig.align_ylabels(axes)
fig.savefig("figs/fig_timeline.pdf"); fig.savefig("figs/fig_timeline.png"); plt.show()

## 4 · The replicate-noise ceiling (Proposition 3.1)
Each reported index is the mean of two same-day replicates, so the dataset
carries its own noise estimate. The robust identity
sd(delta) = sqrt(pi/2) * E|delta| guards against clerical outliers; the
ceiling and the oracle-MAE floor follow directly.

In [ ]:
NOISE = {}
for t in TARGETS:
    lo_b, hi_b = TARGET_BOUNDS[t]
    a = REPS[f"{t}_a"].where(REPS[f"{t}_a"].between(lo_b, hi_b))
    b = REPS[f"{t}_b"].where(REPS[f"{t}_b"].between(lo_b, hi_b))
    both = np.isfinite(a) & np.isfinite(b)
    d = (a[both] - b[both]).values
    mean_abs = float(np.abs(d).mean())
    sd_e = mean_abs * np.sqrt(np.pi / 2) / np.sqrt(2)
    var_mean = sd_e ** 2 / 2
    _, _, y, _, cut = per_target(t)
    NOISE[t] = {
        "n_pairs": int(both.sum()), "mean_abs_diff": mean_abs,
        "sd_noise_mean": float(np.sqrt(var_mean)),
        "oracle_mae": float(np.sqrt(var_mean) * np.sqrt(2 / np.pi)),
        "rep_corr": float(np.corrcoef(a[both], b[both])[0, 1]),
        "r2_ceiling_all": float(1 - var_mean / np.var(y, ddof=1)),
        "r2_ceiling_test": float(1 - var_mean / np.var(y[cut:], ddof=1)),
        "var_train": float(np.var(y[:cut], ddof=1)),
        "var_test": float(np.var(y[cut:], ddof=1))}
    v = NOISE[t]
    print(f"[{t}] pairs={v['n_pairs']}  E|d|={mean_abs:.3f}  "
          f"sd(mean)={v['sd_noise_mean']:.3f}  oracleMAE={v['oracle_mae']:.3f}  "
          f"R2ceil(all)={v['r2_ceiling_all']:.3f}  R2ceil(test)={v['r2_ceiling_test']:.3f}")
json.dump(NOISE, open("models/noise.json", "w"), indent=1)

## 5 · The sixteen-model zoo under both protocols
Identical pipelines (kNN imputation + standardization fitted inside every
fold) behind sixteen learners. P-CV is shuffled repeated K-fold, the protocol
the application literature uses; P-CHRON trains on the first 80% and tests on
the final 250 days, the protocol a deployed sensor experiences. Three naive
predictors that use no same-day measurement join the chronological block.
*This is the long cell: roughly 25 minutes with `FULL_RUN = True`.*

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, AdaBoostRegressor,
                              VotingRegressor, StackingRegressor)
from sklearn.model_selection import KFold, RepeatedKFold, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb

RS = RANDOM_STATE

def make_preprocessor():
    return [("imputer", KNNImputer(n_neighbors=5)), ("scaler", StandardScaler())]

def pipe(est, extra=None):
    return Pipeline(make_preprocessor() + (extra or []) + [("model", est)])

def build_models():
    rf = RandomForestRegressor(n_estimators=200, random_state=RS, n_jobs=-1)
    et = ExtraTreesRegressor(n_estimators=200, random_state=RS, n_jobs=-1)
    xg = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.9, colsample_bytree=0.9,
                          random_state=RS, n_jobs=-1, verbosity=0)
    base = [("rf", RandomForestRegressor(n_estimators=150, random_state=RS, n_jobs=-1)),
            ("et", ExtraTreesRegressor(n_estimators=150, random_state=RS, n_jobs=-1)),
            ("xg", xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                    random_state=RS, n_jobs=-1, verbosity=0))]
    return {
        "Linear Regression": pipe(LinearRegression()),
        "Ridge": pipe(Ridge(alpha=10.0)),
        "Lasso": pipe(Lasso(alpha=0.01, max_iter=5000)),
        "ElasticNet": pipe(ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000)),
        "Polynomial(2)+Ridge": pipe(Ridge(alpha=10.0),
                                    extra=[("poly", PolynomialFeatures(2, include_bias=False))]),
        "SVR (RBF)": pipe(SVR(C=10, gamma="scale")),
        "KNN": pipe(KNeighborsRegressor(n_neighbors=8, weights="distance")),
        "Decision Tree": pipe(DecisionTreeRegressor(max_depth=8, random_state=RS)),
        "MLP": pipe(MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000,
                                 early_stopping=True, random_state=RS)),
        "Random Forest": pipe(rf),
        "Extra Trees": pipe(et),
        "Gradient Boosting": pipe(GradientBoostingRegressor(random_state=RS)),
        "AdaBoost": pipe(AdaBoostRegressor(n_estimators=200, learning_rate=0.5, random_state=RS)),
        "XGBoost": pipe(xg),
        "Voting Ensemble": pipe(VotingRegressor(base)),
        "Stacking Ensemble": pipe(StackingRegressor(estimators=base,
                                                    final_estimator=Ridge(alpha=1.0), cv=3)),
    }
print(len(build_models()), "models defined")

In [ ]:
RES = {t: {"cv_summary": {}, "cv_folds": {}, "chrono": {}, "naive": {}} for t in TARGETS}
t0 = time.time()
for t in TARGETS:
    _, X, y, _, i80 = per_target(t)
    for name, model in build_models().items():
        cv = cross_validate(model, X, y,
                            cv=RepeatedKFold(n_splits=5, n_repeats=N_REPEATS,
                                             random_state=RANDOM_STATE),
                            scoring={"R2": "r2", "MAE": "neg_mean_absolute_error"},
                            n_jobs=1)
        RES[t]["cv_summary"][name] = {
            "R2_mean": cv["test_R2"].mean(), "R2_std": cv["test_R2"].std(ddof=1),
            "MAE_mean": -cv["test_MAE"].mean()}
        RES[t]["cv_folds"][name] = cv["test_R2"]
        m = build_models()[name]
        m.fit(X.iloc[:i80], y[:i80]); yp = m.predict(X.iloc[i80:])
        RES[t]["chrono"][name] = {"R2": r2_score(y[i80:], yp),
                                  "MAE": mean_absolute_error(y[i80:], yp)}
        s = RES[t]["cv_summary"][name]
        print(f"[{t}] {name:22s} CV R2={s['R2_mean']:+.3f}±{s['R2_std']:.3f} "
              f"MAE={s['MAE_mean']:.3f} | chrono R2={RES[t]['chrono'][name]['R2']:+.3f} "
              f"MAE={RES[t]['chrono'][name]['MAE']:.3f}", flush=True)
    yte = y[i80:]
    RES[t]["naive"]["Mean"] = {"R2": r2_score(yte, np.full_like(yte, y[:i80].mean())),
                               "MAE": mean_absolute_error(yte, np.full_like(yte, y[:i80].mean()))}
    lag1 = pd.Series(y).shift(1).values[i80:]
    roll7 = pd.Series(y).shift(1).rolling(7, min_periods=1).mean().values[i80:]
    RES[t]["naive"]["Persistence"] = {"R2": r2_score(yte, lag1),
                                      "MAE": mean_absolute_error(yte, lag1)}
    RES[t]["naive"]["Rolling-7"] = {"R2": r2_score(yte, roll7),
                                    "MAE": mean_absolute_error(yte, roll7)}
    for k, v in RES[t]["naive"].items():
        print(f"[{t}] naive {k:12s} chrono R2={v['R2']:+.3f} MAE={v['MAE']:.3f}")
print(f"benchmark done in {time.time()-t0:.0f}s")

Significance across the CV folds: Friedman, then one-sided Wilcoxon
signed-rank against the leader under Holm correction. The honest reading is a
statistical tie among several classical learners, with the perceptron failing
outright.

In [ ]:
SIG = {}
for t in TARGETS:
    folds = RES[t]["cv_folds"]; names = list(folds)
    fr = stats.friedmanchisquare(*[folds[nm] for nm in names])
    best = max(names, key=lambda nm: folds[nm].mean())
    pv = {nm: stats.wilcoxon(folds[best], folds[nm], alternative="greater").pvalue
          for nm in names if nm != best}
    order = sorted(pv, key=pv.get); m = len(order)
    holm = {nm: min(1.0, pv[nm] * (m - i)) for i, nm in enumerate(order)}
    ties = [nm for nm in names if nm != best and holm[nm] > 0.05]
    SIG[t] = {"best": best, "chi2": fr.statistic, "p": fr.pvalue, "ties": ties}
    print(f"[{t}] best={best}  Friedman chi2={fr.statistic:.1f} p={fr.pvalue:.2e}  "
          f"indistinguishable: {ties}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.75))
for ax, t in zip(axes, TARGETS):
    s = RES[t]["cv_summary"]
    names = sorted(s, key=lambda k: s[k]["R2_mean"])
    means = [s[k]["R2_mean"] for k in names]; stds = [s[k]["R2_std"] for k in names]
    colors = ["#C0392B" if "Voting" in k else ("#777" if s[k]["R2_mean"] < 0 else COL[t])
              for k in names]
    ax.barh(range(len(names)), means, xerr=stds, color=colors, height=.62,
            error_kw={"lw": .5, "capsize": 1.4})
    ax.set_yticks(range(len(names)), names)
    ax.axvline(0, color="k", lw=.7)
    ax.set_xlabel(r"cross-validated $R^2$ (5$\times$%d folds)" % N_REPEATS)
    ax.set_title(t, fontsize=8)
    ax.set_xlim(max(min(means) - .05, -1.0), max(means) + .12)
fig.tight_layout(w_pad=1.4)
fig.savefig("figs/fig_cv.pdf"); fig.savefig("figs/fig_cv.png"); plt.show()

## 6 · The hybrid sensor
Two lagged laboratory terms, the last confirmed value and its seven-day mean,
join the eleven physical features. Every hybrid head turns the chronological
$R^2$ positive; the ridge head generalizes best under drift, and **Figure 4**
sets the whole progression against the dotted oracle floors.

In [ ]:
HYB = {}
for t in TARGETS:
    d2 = df.copy(); y0 = d2[t].values
    d2["y_lag1"] = pd.Series(y0).shift(1)
    d2["y_roll7"] = pd.Series(y0).shift(1).rolling(7, min_periods=1).mean()
    d2 = d2.iloc[1:].reset_index(drop=True)
    y = d2[t].values; nh = len(y); h80 = int(round(.8 * nh)); h60 = int(round(.6 * nh))
    HYBF = FINAL_FEATURES + ["y_lag1", "y_roll7"]
    Xh, Xa = d2[HYBF], d2[["y_lag1", "y_roll7"]]
    HYB[t] = {"d": d2, "y": y, "h60": h60, "h80": h80, "HYBF": HYBF}
    for vname, (mname, Xv) in {"Hybrid Voting": ("Voting Ensemble", Xh),
                               "Hybrid KNN": ("KNN", Xh),
                               "Hybrid Ridge": ("Ridge", Xh),
                               "AR-only Ridge": ("Ridge", Xa)}.items():
        m = build_models()[mname]; m.fit(Xv.iloc[:h80], y[:h80])
        yp = m.predict(Xv.iloc[h80:])
        cv = cross_validate(build_models()[mname], Xv, y,
                            cv=RepeatedKFold(n_splits=5, n_repeats=N_REPEATS,
                                             random_state=RANDOM_STATE),
                            scoring={"R2": "r2", "MAE": "neg_mean_absolute_error"}, n_jobs=1)
        HYB[t][vname] = {"R2": r2_score(y[h80:], yp),
                         "MAE": mean_absolute_error(y[h80:], yp),
                         "CV_R2": cv["test_R2"].mean(), "CV_R2_sd": cv["test_R2"].std(ddof=1),
                         "CV_MAE": -cv["test_MAE"].mean(), "pred": yp}
        v = HYB[t][vname]
        print(f"[{t}] {vname:14s} chrono R2={v['R2']:+.3f} MAE={v['MAE']:.3f} | "
              f"CV R2={v['CV_R2']:+.3f}±{v['CV_R2_sd']:.3f}", flush=True)

In [ ]:
fig, ax = plt.subplots(figsize=(3.34, 2.0))
groups = ["mean", "best\nfeature-only", "persist.", "roll-7", "AR-only", "hybrid"]
width = .38
for gi, t in enumerate(TARGETS):
    best_feat = min(RES[t]["chrono"], key=lambda k: RES[t]["chrono"][k]["MAE"])
    vals = [RES[t]["naive"]["Mean"]["MAE"], RES[t]["chrono"][best_feat]["MAE"],
            RES[t]["naive"]["Persistence"]["MAE"], RES[t]["naive"]["Rolling-7"]["MAE"],
            HYB[t]["AR-only Ridge"]["MAE"], HYB[t]["Hybrid Ridge"]["MAE"]]
    ax.bar(np.arange(len(groups)) + (gi - .5) * width, vals, width, label=t,
           color=COL[t], alpha=.9)
for t in TARGETS:
    ax.axhline(NOISE[t]["oracle_mae"], color=COL[t], lw=.8, ls=":", alpha=.9)
ax.set_xticks(range(len(groups)), groups, fontsize=6.2)
ax.set_ylabel("chronological test MAE")
ax.annotate("dotted: replicate-noise floor", (0.99, 0.97), xycoords="axes fraction",
            ha="right", va="top", fontsize=5.8, color="#444")
ax.legend(frameon=False, loc="upper left")
fig.savefig("figs/fig_chrono_mae.pdf"); fig.savefig("figs/fig_chrono_mae.png"); plt.show()

## 7 · Uncertainty that survives drift
Three calibration regimes on identical predictions: a static split whose
calibration window precedes the test years, a five-fold cross-conformal pool,
and the rolling recalibration of Eq. (5) with W = 100. The rolling scheme is
the one that is both valid and narrow, and it draws **Figure 5**.

In [ ]:
def q_of(scores, alpha):
    s = np.sort(np.asarray(scores)); m = len(s)
    return s[min(ceil((m + 1) * (1 - alpha)), m) - 1]

CONF = {}
for t in TARGETS:
    d2, y = HYB[t]["d"], HYB[t]["y"]
    h60, h80, HYBF = HYB[t]["h60"], HYB[t]["h80"], HYB[t]["HYBF"]
    Xh = d2[HYBF]
    CONF[t] = {}
    # hybrid Voting: static split and cross (paper Table 4, HY rows)
    m = build_models()["Voting Ensemble"]; m.fit(Xh.iloc[:h60], y[:h60])
    s_cal = np.abs(y[h60:h80] - m.predict(Xh.iloc[h60:h80]))
    yp = m.predict(Xh.iloc[h80:])
    for a in (0.2, 0.1):
        q = q_of(s_cal, a)
        CONF[t][f"HY split {int((1-a)*100)}"] = {
            "cov": float(np.mean(np.abs(y[h80:] - yp) <= q)), "width": float(2 * q)}
    kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
    sc = np.empty(h80)
    for tr, va in kf.split(Xh.iloc[:h80]):
        mm = build_models()["Voting Ensemble"]
        mm.fit(Xh.iloc[:h80].iloc[tr], y[:h80][tr])
        sc[va] = np.abs(y[:h80][va] - mm.predict(Xh.iloc[:h80].iloc[va]))
    mm = build_models()["Voting Ensemble"]; mm.fit(Xh.iloc[:h80], y[:h80])
    yp2 = mm.predict(Xh.iloc[h80:])
    for a in (0.2, 0.1):
        q = q_of(sc, a)
        CONF[t][f"HY cross {int((1-a)*100)}"] = {
            "cov": float(np.mean(np.abs(y[h80:] - yp2) <= q)), "width": float(2 * q)}
    # rolling on hybrid Ridge (the deployed configuration)
    mr = build_models()["Ridge"]; mr.fit(Xh.iloc[:h60], y[:h60])
    resid = np.abs(y[h60:] - mr.predict(Xh.iloc[h60:]))
    n_cal = h80 - h60
    ROLLPRED = mr.predict(Xh.iloc[h80:])
    for a in (0.2, 0.1):
        qs = np.array([q_of(resid[max(0, j - W):j], a)
                       for j in range(n_cal, len(resid))])
        cov = np.mean(resid[n_cal:] <= qs)
        CONF[t][f"rolling {int((1-a)*100)}"] = {"cov": float(cov),
                                                "width": float(2 * qs.mean()),
                                                "q_series": qs}
    for k, v in CONF[t].items():
        print(f"[{t}] {k:12s} cov={v['cov']:.3f} width={v['width']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7.0, 3.0))
for ax, t in zip(axes, TARGETS):
    d2, y = HYB[t]["d"], HYB[t]["y"]
    h60, h80 = HYB[t]["h60"], HYB[t]["h80"]
    dts = d2["Date"].values[h80:]; yt = y[h80:]
    mr = build_models()["Ridge"]
    mr.fit(d2[HYB[t]["HYBF"]].iloc[:h60], y[:h60])
    yp = mr.predict(d2[HYB[t]["HYBF"]].iloc[h80:])
    qs = CONF[t]["rolling 90"]["q_series"]; cov = CONF[t]["rolling 90"]["cov"]
    ax.fill_between(dts, yp - qs, yp + qs, color=COL[t], alpha=.18, lw=0,
                    label=f"rolling 90% conformal band (mean width "
                          f"{CONF[t]['rolling 90']['width']:.2f})")
    ax.plot(dts, yp, "-", color=COL[t], lw=.8, label="hybrid prediction")
    inside = np.abs(yt - yp) <= qs
    ax.plot(dts[inside], yt[inside], ".", ms=2.4, color="#333", label="lab value (covered)")
    ax.plot(dts[~inside], yt[~inside], "x", ms=3.2, color="#C0392B", mew=1.0,
            label="lab value (missed)")
    ax.set_ylabel(t)
    ax.annotate(f"empirical coverage {cov:.1%}", (0.01, 0.04),
                xycoords="axes fraction", fontsize=6.4, color="#333")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    if t == "CRI":
        ax.legend(ncols=4, frameon=False, loc="upper left",
                  bbox_to_anchor=(0, 1.24), columnspacing=1.0, handletextpad=.4)
axes[1].set_xlabel("date (chronological test block)")
fig.align_ylabels(axes); fig.tight_layout(h_pad=.9)
fig.savefig("figs/fig_conformal.pdf"); fig.savefig("figs/fig_conformal.png"); plt.show()

## 8 · Ablation and attribution
Feature-set ablation on the voting head (Table 5 of the paper) plus
permutation importance on the chronological block (**Figure 6**). Coal-side
features from the blend sheet are merged by date to show that a blend held to
a 0.7-point ash interquartile range carries essentially no signal.

In [ ]:
# coal-side daily features from the 'Blend coal' sheet (mirrors the paper pipeline)
rawb = pd.read_excel(DATA_PATH, sheet_name="Blend coal", header=None)
coal = pd.DataFrame({
    "Date": pd.to_datetime(rawb.iloc[8:, 0], errors="coerce").reset_index(drop=True),
    "Coal_Ash": pd.to_numeric(rawb.iloc[8:, 3], errors="coerce").reset_index(drop=True),
    "Coal_VM": pd.to_numeric(rawb.iloc[8:, 6], errors="coerce").reset_index(drop=True),
    "Coal_FC": pd.to_numeric(rawb.iloc[8:, 9], errors="coerce").reset_index(drop=True),
    "Coal_WM": pd.to_numeric(rawb.iloc[8:, 12], errors="coerce").reset_index(drop=True),
    "Coal_m3.2mm": pd.to_numeric(rawb.iloc[8:, 15], errors="coerce").reset_index(drop=True),
}).dropna(subset=["Date"])
coal["Date"] = coal["Date"].dt.normalize()
coal = coal.groupby("Date", as_index=False).mean()
COAL_FEATS = ["Coal_Ash", "Coal_VM", "Coal_FC", "Coal_WM", "Coal_m3.2mm"]
print("coal rows:", len(coal), "| Coal_Ash IQR:",
      round(coal.Coal_Ash.quantile(.25), 2), round(coal.Coal_Ash.quantile(.75), 2))
merged = df.merge(coal, on="Date", how="inner")
print("merged rows:", len(merged))

ABLA = {}
SETS = {"full (8 raw + 3 eng.)": FINAL_FEATURES,
        "raw only (8)": RAW_FEATURES,
        "size + M40 (3)": ["M40", "Coke_+80mm", "Coke_-25mm"],
        "coal-side only (5)": COAL_FEATS}
for t in TARGETS:
    ABLA[t] = {}
    for k, cols_ in SETS.items():
        src_ = merged if k.startswith("coal") else df
        d_t = src_.dropna(subset=[t])
        cv = cross_validate(build_models()["Voting Ensemble"], d_t[cols_], d_t[t].values,
                            cv=RepeatedKFold(n_splits=5, n_repeats=N_REPEATS,
                                             random_state=RANDOM_STATE),
                            scoring={"R2": "r2", "MAE": "neg_mean_absolute_error"}, n_jobs=1)
        ABLA[t][k] = {"n": len(d_t), "R2": cv["test_R2"].mean(),
                      "sd": cv["test_R2"].std(ddof=1), "MAE": -cv["test_MAE"].mean()}
        print(f"[{t}] {k:22s} n={len(d_t):4d} R2={ABLA[t][k]['R2']:+.3f}"
              f"±{ABLA[t][k]['sd']:.3f} MAE={ABLA[t][k]['MAE']:.3f}", flush=True)

In [ ]:
from sklearn.inspection import permutation_importance
fig, axes = plt.subplots(2, 1, figsize=(3.34, 3.1))
pretty = {"Coke_+80mm": "+80 mm", "Coke_-25mm": "$-$25 mm", "Coke_WM": "moisture",
          "Coke_Ash": "ash", "Coke_VM": "VM", "Coke_FC": "FC", "M40": "M40",
          "M10": "M10", "FC_Ash_ratio": "FC/ash", "Strength_idx": "M40$-$M10",
          "Ash_VM": "ash$\\times$VM"}
for ax, t in zip(axes, TARGETS):
    _, X, y, _, i80 = per_target(t)
    m = build_models()["Voting Ensemble"]; m.fit(X.iloc[:i80], y[:i80])
    pi = permutation_importance(m, X.iloc[i80:], y[i80:], n_repeats=N_PERM,
                                random_state=RANDOM_STATE,
                                scoring="neg_mean_absolute_error")
    order = np.argsort(pi.importances_mean)[::-1][:8][::-1]
    ax.barh(range(len(order)), pi.importances_mean[order],
            xerr=pi.importances_std[order], color=COL[t], height=.6,
            error_kw={"lw": .5, "capsize": 1.4})
    ax.set_yticks(range(len(order)), [pretty[FINAL_FEATURES[i]] for i in order])
    ax.set_title(t, fontsize=7.5, loc="left")
axes[1].set_xlabel("permutation importance (MAE increase)")
fig.tight_layout(h_pad=.8)
fig.savefig("figs/fig_importance.pdf"); fig.savefig("figs/fig_importance.png"); plt.show()

## 9 · The edge budget
Artifact size against single-sample latency on the current machine
(**Figure 7**). Sizes reproduce exactly; latencies scale with the host CPU,
and the paper's figures come from one Intel Xeon core at 2.8 GHz.

In [ ]:
import io
LAT = {}
BENCH = ["KNN", "Voting Ensemble", "Random Forest", "Extra Trees", "XGBoost",
         "Ridge", "SVR (RBF)", "Gradient Boosting", "MLP", "Stacking Ensemble"]
for t in TARGETS:
    LAT[t] = {}
    _, X, y, _, _ = per_target(t)
    for name in BENCH:
        m = build_models()[name]; m.fit(X, y)          # full-data fit, as deployed
        buf = io.BytesIO(); joblib.dump(m, buf, compress=6)
        size_kb = buf.getbuffer().nbytes / 1024
        row = X.iloc[[0]]
        for _ in range(10): m.predict(row)
        ts = []
        for _ in range(N_TIMED):
            a = time.perf_counter(); m.predict(row); ts.append(time.perf_counter() - a)
        ts = np.array(ts) * 1e3
        tb = time.perf_counter(); m.predict(X.iloc[:256])
        batch_ms = (time.perf_counter() - tb) * 1e3
        LAT[t][name] = {"kb": size_kb, "ms": float(np.median(ts)),
                        "p95": float(np.percentile(ts, 95)), "batch256": batch_ms}
        print(f"[{t}] {name:18s} {size_kb:8.1f} KB  med={np.median(ts):5.2f} ms  "
              f"p95={np.percentile(ts,95):5.2f} ms  batch256={batch_ms:6.1f} ms", flush=True)

fig, ax = plt.subplots(figsize=(3.34, 2.25))
for t, mk in zip(TARGETS, ("o", "s")):
    for name, v in LAT[t].items():
        ax.scatter(v["kb"], v["ms"], s=13, marker=mk, color=COL[t],
                   alpha=.85, edgecolors="none")
        if t == "CRI" and name in ("KNN", "Voting Ensemble", "Ridge", "Stacking Ensemble"):
            lbl = {"Voting Ensemble": "Voting", "Stacking Ensemble": "Stacking"}.get(name, name)
            ax.annotate(lbl, (v["kb"], v["ms"]), textcoords="offset points",
                        xytext=(4, 3), fontsize=6)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("serialized model size (KB)"); ax.set_ylabel("single-sample latency (ms)")
for t, mk in zip(TARGETS, ("o", "s")):
    ax.scatter([], [], marker=mk, color=COL[t], s=13, label=t)
ax.legend(frameon=False, loc="upper left")
fig.savefig("figs/fig_latency.pdf"); fig.savefig("figs/fig_latency.png"); plt.show()

## 10 · Paper-number audit
With `FULL_RUN = True`, every value below matches the manuscript to the
printed precision; with the fast setting, small deviations in the CV columns
are expected because only one repeat is run. Latency rows are excluded by
design (machine-dependent).

In [ ]:
EXPECT = {
 ("CRI", "rows"): (N_T["CRI"], 1251, 0), ("CSR", "rows"): (N_T["CSR"], 1251, 0),
 ("CRI", "pairs"): (NOISE["CRI"]["n_pairs"], 793, 0),
 ("CSR", "pairs"): (NOISE["CSR"]["n_pairs"], 792, 0),
 ("CRI", "E|d|"): (NOISE["CRI"]["mean_abs_diff"], 0.841, .01),
 ("CSR", "E|d|"): (NOISE["CSR"]["mean_abs_diff"], 1.183, .01),
 ("CRI", "oracle MAE"): (NOISE["CRI"]["oracle_mae"], 0.420, .01),
 ("CSR", "oracle MAE"): (NOISE["CSR"]["oracle_mae"], 0.592, .01),
 ("CRI", "R2 ceil test"): (NOISE["CRI"]["r2_ceiling_test"], 0.482, .02),
 ("CSR", "R2 ceil test"): (NOISE["CSR"]["r2_ceiling_test"], 0.554, .02),
 ("CRI", "Voting CV R2"): (RES["CRI"]["cv_summary"]["Voting Ensemble"]["R2_mean"], 0.303, .03),
 ("CSR", "Voting CV R2"): (RES["CSR"]["cv_summary"]["Voting Ensemble"]["R2_mean"], 0.398, .03),
 ("CRI", "Voting CV MAE"): (RES["CRI"]["cv_summary"]["Voting Ensemble"]["MAE_mean"], 0.713, .02),
 ("CSR", "Voting CV MAE"): (RES["CSR"]["cv_summary"]["Voting Ensemble"]["MAE_mean"], 1.066, .02),
 ("CRI", "roll-7 MAE"): (RES["CRI"]["naive"]["Rolling-7"]["MAE"], 0.457, .01),
 ("CSR", "roll-7 MAE"): (RES["CSR"]["naive"]["Rolling-7"]["MAE"], 0.683, .01),
 ("CRI", "hybrid ridge MAE"): (HYB["CRI"]["Hybrid Ridge"]["MAE"], 0.466, .01),
 ("CSR", "hybrid ridge MAE"): (HYB["CSR"]["Hybrid Ridge"]["MAE"], 0.627, .01),
 ("CRI", "hybrid ridge R2"): (HYB["CRI"]["Hybrid Ridge"]["R2"], 0.31, .03),
 ("CSR", "hybrid ridge R2"): (HYB["CSR"]["Hybrid Ridge"]["R2"], 0.42, .03),
 ("CRI", "rolling cov 90"): (CONF["CRI"]["rolling 90"]["cov"], 0.904, .01),
 ("CSR", "rolling cov 90"): (CONF["CSR"]["rolling 90"]["cov"], 0.924, .01),
 ("CRI", "rolling width 90"): (CONF["CRI"]["rolling 90"]["width"], 2.01, .05),
 ("CSR", "rolling width 90"): (CONF["CSR"]["rolling 90"]["width"], 3.18, .05),
}
fails = 0
for (t, k), (got, exp, tol) in EXPECT.items():
    ok = abs(got - exp) <= max(tol, 1e-9)
    fails += (not ok)
    print(f"{'PASS' if ok else 'FAIL':4s}  {t} {k:18s} paper={exp}  notebook={got:.3f}")
print(("\nAll audited numbers match the manuscript."
       if fails == 0 else f"\n{fails} mismatches - expected under FULL_RUN=False "
       "for CV rows; investigate otherwise."))

## 11 · Deployment artifacts and download
The evaluated deployable of the paper: hybrid ridge fitted on the first 80%,
rolling buffer seeded with the last 100 replay residuals, quantiles per
Eq. (5). These are the exact files the Streamlit dashboard loads, followed by
a zip of figures and models for download.

In [ ]:
def q_of(scores, alpha):                      # self-contained for reuse
    s = np.sort(np.asarray(scores)); m = len(s)
    return s[min(ceil((m + 1) * (1 - alpha)), m) - 1]

DEPLOY_META = {"system": "CokeSense",
               "model": "hybrid ridge (paper Table 3, deployable row)",
               "features": None, "window_W": W, "sklearn": sklearn.__version__,
               "trained_on": "first 80% of timeline", "targets": {}}
for t in TARGETS:
    d2, y = HYB[t]["d"], HYB[t]["y"]
    h60, h80, HYBF = HYB[t]["h60"], HYB[t]["h80"], HYB[t]["HYBF"]
    m = build_models()["Ridge"]; m.fit(d2[HYBF].iloc[:h80], y[:h80])
    joblib.dump(m, f"models/hybrid_ridge_{t}.joblib", compress=3)
    m60 = build_models()["Ridge"]; m60.fit(d2[HYBF].iloc[:h60], y[:h60])
    resid = np.abs(y[h60:] - m60.predict(d2[HYBF].iloc[h60:]))
    buf = resid[-W:].tolist()
    last = d2.iloc[-1]
    DEPLOY_META["targets"][t] = {
        "q80": float(q_of(np.array(buf), 0.2)), "q90": float(q_of(np.array(buf), 0.1)),
        "buffer_tail": buf, "default_lag1": float(last["y_lag1"]),
        "default_roll7": float(last["y_roll7"]),
        "last_date": str(pd.Timestamp(last["Date"]).date()),
        "train_rows": int(h80),
        "spec_note": "paper coverage at 90%: CRI 90.4%, CSR 92.4%"}
    print(f"[{t}] saved models/hybrid_ridge_{t}.joblib  "
          f"q90={DEPLOY_META['targets'][t]['q90']:.2f}")
DEPLOY_META["features"] = FINAL_FEATURES + ["y_lag1", "y_roll7"]
DEPLOY_META["raw_feature_ranges"] = {f: [float(np.nanpercentile(df[f], 1)),
                                         float(np.nanpercentile(df[f], 99))]
                                     for f in RAW_FEATURES}
DEPLOY_META["raw_feature_defaults"] = {f: float(df[f].median()) for f in RAW_FEATURES}
json.dump(DEPLOY_META, open("models/meta.json", "w"), indent=1)
print("models/meta.json written")

import shutil
shutil.make_archive("cokesense_outputs", "zip", root_dir=".",
                    base_dir="figs")
with __import__("zipfile").ZipFile("cokesense_outputs.zip", "a") as z:
    for f in os.listdir("models"):
        z.write(os.path.join("models", f))
print("cokesense_outputs.zip ready (figs/ + models/)")
try:
    from google.colab import files
    files.download("cokesense_outputs.zip")
except Exception:
    pass

---
### Closing notes
* Random seeds fix fold assignment, forests, boosting, and permutation
  importance; the pinned scikit-learn 1.8.0 matters because `RepeatedKFold`
  became keyword-only and because the joblib artifacts load only under the
  training version.
* Figure files carry the exact names the manuscript's `\paperfig` macro
  expects; drop the six PDFs into `manuscript/figs/` and recompile.
* Repository: `github.com/parichay-india/CokeSense` · dashboard:
  `cokequalitybsl.streamlit.app` (both withheld from the anonymized PDF until
  camera-ready).